## 0. Notebook Objective & Scope
Establish a reproducible, leakage-safe data transformation and evaluation protocol for the four Track A cardiovascular prediction tasks (Cath, LAD, LCX, RCA). This notebook defines deterministic cleaning, semantic schema partitioning, two candidate preprocessing pipelines (56 vs 59 features), target-specific repeated stratified cross-validation splitters, and holdout governance without fitting predictive models.

## 1. Imports & Reproducibility

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', None)

## 2. Load Audited Dataset & Reconstruct Frozen Contract
Load raw patient records and verify alignment with the locked findings from Notebook 1.

In [2]:
DATA_PATH = Path('extention of Z-Alizadeh sani dataset.xlsx')
df_raw = pd.read_excel(DATA_PATH)
df = df_raw.copy()

TARGET_COLUMNS = ['Cath', 'LAD', 'LCX', 'RCA']
DROP_COLUMNS = ['Exertional CP']

assert set(TARGET_COLUMNS).issubset(df.columns), "Target columns missing from raw dataset"
assert set(DROP_COLUMNS).issubset(df.columns), "Candidate dropped column missing from raw dataset"
print(f"Loaded raw dataset: {df.shape[0]} rows, {df.shape[1]} columns. Data contract verified.")

Loaded raw dataset: 303 rows, 59 columns. Data contract verified.


## 3. Deterministic Data Cleaning
Apply deterministic string normalizations and drop uninformative constant features without learning any parameters.

In [3]:
# 3.1 Normalize 'Fmale' typographical error in Sex
df['Sex'] = df['Sex'].replace({'Fmale': 'Female'})
assert set(df['Sex'].unique()) == {'Male', 'Female'}, "Sex column contains unexpected categories"

# 3.2 Drop zero-variance constant feature
df = df.drop(columns=DROP_COLUMNS)
assert 'Exertional CP' not in df.columns, "Exertional CP failed to drop"

# 3.3 Validate domain categories against allowed sets
EXPECTED_CATEGORIES = {
    'Sex': {'Male', 'Female'},
    'BBB': {'N', 'LBBB', 'RBBB'},
    'VHD': {'N', 'mild', 'Moderate', 'Severe'},
    'Obesity': {'Y', 'N'},
    'CRF': {'Y', 'N'},
    'CVA': {'Y', 'N'},
    'Airway disease': {'Y', 'N'},
    'Thyroid Disease': {'Y', 'N'},
    'CHF': {'Y', 'N'},
    'DLP': {'Y', 'N'}
}

for col, allowed in EXPECTED_CATEGORIES.items():
    observed = set(df[col].dropna().unique())
    unexpected = observed - allowed
    assert not unexpected, f"Unexpected categories in {col}: {unexpected}"

print(f"Deterministic cleaning complete. Remaining columns: {df.shape[1]}")

Deterministic cleaning complete. Remaining columns: 58


## 4. Target Construction & Encoding Verification
Map raw clinical diagnoses to standard binary indicators (1 = Disease/Stenotic, 0 = Normal) and isolate from feature matrix.

In [4]:
TARGET_MAPPINGS = {
    'Cath': {'CAD': 1, 'Normal': 0},
    'LAD': {'Stenotic': 1, 'Normal': 0},
    'LCX': {'Stenotic': 1, 'Normal': 0},
    'RCA': {'Stenotic': 1, 'Normal': 0}
}

y_df = pd.DataFrame(index=df.index)
for target, mapping in TARGET_MAPPINGS.items():
    observed_vals = set(df[target].unique())
    expected_vals = set(mapping.keys())
    assert observed_vals == expected_vals, f"Target mismatch for {target}: {observed_vals} vs {expected_vals}"
    y_df[target] = df[target].map(mapping).astype(int)
    assert set(y_df[target].unique()) <= {0, 1}, f"Non-binary encoding in {target}"

# Isolate feature matrix
X_raw = df.drop(columns=TARGET_COLUMNS).copy()

print("Target encoding summary:")
display(pd.DataFrame({
    'Target': list(TARGET_MAPPINGS.keys()),
    'Positive (1)': [TARGET_MAPPINGS[k].get('CAD', 'Stenotic') for k in TARGET_MAPPINGS],
    'Positive Count': [y_df[t].sum() for t in TARGET_MAPPINGS],
    'Positive %': [(y_df[t].mean() * 100).round(2) for t in TARGET_MAPPINGS],
    'Negative Count': [(len(y_df) - y_df[t].sum()) for t in TARGET_MAPPINGS]
}))

Target encoding summary:


,Target,Positive (1),Positive Count,Positive %,Negative Count
0,Cath,1,216,71.29,87
1,LAD,Stenotic,177,58.42,126
2,LCX,Stenotic,119,39.27,184
3,RCA,Stenotic,114,37.62,189


## 5. Feature Groups & Semantic Schema
Partition candidate input features into mutually exclusive, exhaustive semantic groups.

In [5]:
# Binary clinical indicators (0/1 numeric flags or Y/N string flags)
BINARY_STR_FEATURES = [
    'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP',
    'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur',
    'Dyspnea', 'Atypical', 'Nonanginal', 'LowTH Ang', 'LVH', 'Poor R Progression'
]
BINARY_NUM_FEATURES = [
    'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'Edema', 
    'Typical Chest Pain', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion'
]
SEX_FEATURE = ['Sex']

ALL_BINARY_FEATURES = BINARY_STR_FEATURES + BINARY_NUM_FEATURES + SEX_FEATURE

# Nominal & Ordinal Categorical
NOMINAL_FEATURES = ['BBB']
ORDINAL_CAT_FEATURES = ['VHD']

# Discrete numerical & continuous physiological features
DISCRETE_NUM_FEATURES = ['Function Class', 'Region RWMA']
CONTINUOUS_FEATURES = [
    'Age', 'Weight', 'Length', 'BMI', 'BP', 'PR', 'FBS', 'CR', 'TG', 'LDL',
    'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE'
]

ALL_NUMERIC_FEATURES = CONTINUOUS_FEATURES + DISCRETE_NUM_FEATURES

ALL_MODEL_FEATURES = ALL_BINARY_FEATURES + NOMINAL_FEATURES + ORDINAL_CAT_FEATURES + ALL_NUMERIC_FEATURES

# Integrity assertions
assert len(ALL_MODEL_FEATURES) == len(set(ALL_MODEL_FEATURES)), "Duplicate features in semantic schema"
assert set(ALL_MODEL_FEATURES) == set(X_raw.columns), "Feature schema does not match cleaned columns"
assert not (set(TARGET_COLUMNS) & set(ALL_MODEL_FEATURES)), "Target leakage detected in feature list"

print(f"Total model features: {len(ALL_MODEL_FEATURES)}")
print(f"  - Binary features:     {len(ALL_BINARY_FEATURES)}")
print(f"  - Nominal features:    {len(NOMINAL_FEATURES)}")
print(f"  - Ordinal features:    {len(ORDINAL_CAT_FEATURES)}")
print(f"  - Numeric features:    {len(ALL_NUMERIC_FEATURES)}")

Total model features: 54
  - Binary features:     29
  - Nominal features:    1
  - Ordinal features:    1
  - Numeric features:    23


## 6. Evaluation Design: Holdout Governance & Support Audit
Audit target balance between training cohort (80%, N=242) and untouched holdout test set (20%, N=61) stratified on primary target Cath. The holdout set remains completely untouched until final evaluation in Notebook 6.

In [6]:
train_idx, holdout_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_df['Cath']
)

split_balance = []
for target in TARGET_COLUMNS:
    full_p = y_df[target].mean() * 100
    tr_p = y_df[target].iloc[train_idx].mean() * 100
    ho_p = y_df[target].iloc[holdout_idx].mean() * 100
    ho_pos = y_df[target].iloc[holdout_idx].sum()
    ho_neg = len(holdout_idx) - ho_pos
    split_balance.append({
        'Target': target,
        'Full Cohort %': round(full_p, 2),
        'Train (N=242) %': round(tr_p, 2),
        'Holdout (N=61) %': round(ho_p, 2),
        'Holdout Pos Count': ho_pos,
        'Holdout Neg Count': ho_neg
    })

split_balance_df = pd.DataFrame(split_balance)
display(split_balance_df)

# Formal holdout governance rules
print("""
HOLDOUT GOVERNANCE RULES:
1. Development Set (N=242): Used exclusively for model comparisons, hyperparameter tuning, and threshold selection.
2. Holdout Set (N=61): Remains strictly untouched and sealed until locked final evaluation in Notebook 6.
3. No model decisions, feature selection, or hyperparameter choices may peek at the holdout.
""")

,Target,Full Cohort %,Train (N=242) %,Holdout (N=61) %,Holdout Pos Count,Holdout Neg Count
0,Cath,71.29,71.49,70.49,43,18
1,LAD,58.42,59.09,55.74,34,27
2,LCX,39.27,38.43,42.62,26,35
3,RCA,37.62,38.84,32.79,20,41



HOLDOUT GOVERNANCE RULES:
1. Development Set (N=242): Used exclusively for model comparisons, hyperparameter tuning, and threshold selection.
2. Holdout Set (N=61): Remains strictly untouched and sealed until locked final evaluation in Notebook 6.
3. No model decisions, feature selection, or hyperparameter choices may peek at the holdout.



## 7. Train / Holdout Partitioning
Partition feature matrix and target series into development and untouched holdout sets.

In [7]:
X_dev = X_raw.iloc[train_idx].copy().reset_index(drop=True)
y_dev = y_df.iloc[train_idx].copy().reset_index(drop=True)

X_holdout = X_raw.iloc[holdout_idx].copy().reset_index(drop=True)
y_holdout = y_df.iloc[holdout_idx].copy().reset_index(drop=True)

print(f"Development Set: {X_dev.shape[0]} rows, {X_dev.shape[1]} features")
print(f"Holdout Set:     {X_holdout.shape[0]} rows, {X_holdout.shape[1]} features")

Development Set: 242 rows, 54 features
Holdout Set:     61 rows, 54 features


## 8. Target-Specific Repeated Stratified K-Fold Strategy
Each of the four prediction tasks (Cath, LAD, LCX, RCA) receives its own independent RepeatedStratifiedKFold splitter (5 folds x 5 repeats = 25 evaluations).

In [8]:
TARGET_CV_SPLITTERS = {
    target: RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)
    for target in TARGET_COLUMNS
}

# Audit CV balance across all 25 folds for each target
cv_summary_records = []
for target in TARGET_COLUMNS:
    splitter = TARGET_CV_SPLITTERS[target]
    val_pos_pcts = []
    
    for tr_i, val_i in splitter.split(X_dev, y_dev[target]):
        assert len(set(tr_i) & set(val_i)) == 0, f"Index overlap in {target} fold"
        val_pos = y_dev[target].iloc[val_i].mean() * 100
        val_pos_pcts.append(val_pos)
        
    cv_summary_records.append({
        'Target': target,
        'Cohort Positive %': round(y_dev[target].mean() * 100, 2),
        'Mean Val Positive %': round(np.mean(val_pos_pcts), 2),
        'Std Val Positive %': round(np.std(val_pos_pcts), 2),
        'Min Val Positive %': round(np.min(val_pos_pcts), 2),
        'Max Val Positive %': round(np.max(val_pos_pcts), 2),
        'Total Evaluations': len(val_pos_pcts)
    })

cv_summary_df = pd.DataFrame(cv_summary_records)
print("Target-Specific Cross-Validation Balance Summary (25 evaluations per target):")
display(cv_summary_df)

Target-Specific Cross-Validation Balance Summary (25 evaluations per target):


,Target,Cohort Positive %,Mean Val Positive %,Std Val Positive %,Min Val Positive %,Max Val Positive %,Total Evaluations
0,Cath,71.49,71.49,0.76,70.83,72.92,25
1,LAD,59.09,59.09,0.76,58.33,60.42,25
2,LCX,38.43,38.43,0.81,37.50,39.58,25
3,RCA,38.84,38.84,0.76,37.50,39.58,25


## 9. Preprocessing Architecture: Encoding Pipelines
Define modular transformers for binary strings, nominal categories, and ordinal severity scales.

In [9]:
# Binary string mapper: Y/N -> 1/0, Male/Female -> 1/0
binary_str_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['N', 'Y']] * len(BINARY_STR_FEATURES),
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

sex_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['Female', 'Male']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Nominal BBB pipeline: OneHotEncoding (N, LBBB, RBBB -> 3 columns)
bbb_pipeline = Pipeline([
    ('ohe', OneHotEncoder(
        categories=[['N', 'LBBB', 'RBBB']],
        handle_unknown='ignore',
        sparse_output=False
    ))
])

# Numeric Standardization pipeline
numeric_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

## 10. VHD Encoding Experiment Setup: Candidate A vs Candidate B
Candidate A uses ordinal encoding (N=0, mild=1, Moderate=2, Severe=3), preserving severity ordering but assuming equal numerical intervals. Candidate B uses nominal One-Hot Encoding without assuming linearity. Downstream modeling in Notebooks 3-5 will systematically evaluate both.

In [10]:
# Candidate A: Ordinal clinical severity (1 output column)
vhd_ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['N', 'mild', 'Moderate', 'Severe']],
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Candidate B: Nominal One-Hot representation (4 output columns)
vhd_onehot_pipeline = Pipeline([
    ('ohe', OneHotEncoder(
        categories=[['N', 'mild', 'Moderate', 'Severe']],
        handle_unknown='ignore',
        sparse_output=False
    ))
])

## 11. ColumnTransformer Pipeline Construction
Assemble full ColumnTransformer pipelines for Candidate A (56 transformed features) and Candidate B (59 transformed features).

In [11]:
def build_preprocessor(vhd_mode='ordinal'):
    vhd_pipe = vhd_ordinal_pipeline if vhd_mode == 'ordinal' else vhd_onehot_pipeline
    
    transformers = [
        ('num', numeric_pipeline, ALL_NUMERIC_FEATURES),
        ('bin_str', binary_str_pipeline, BINARY_STR_FEATURES),
        ('sex', sex_pipeline, SEX_FEATURE),
        ('bbb', bbb_pipeline, NOMINAL_FEATURES),
        ('vhd', vhd_pipe, ORDINAL_CAT_FEATURES),
        ('bin_num', 'passthrough', BINARY_NUM_FEATURES)
    ]
    
    return ColumnTransformer(transformers=transformers, remainder='drop')

preprocessor_candidate_a = build_preprocessor(vhd_mode='ordinal')
preprocessor_candidate_b = build_preprocessor(vhd_mode='onehot')

print("Preprocessing Configuration A (Ordinal VHD) and Configuration B (One-Hot VHD) successfully built.")

Preprocessing Configuration A (Ordinal VHD) and Configuration B (One-Hot VHD) successfully built.


## 12. Preprocessing Isolation & Leakage Verification
Verify transformer fitting strictly on development data and transform-only on holdout data. Fold-level preprocessing isolation will be strictly enforced through sklearn model Pipelines in Notebooks 3-5.

In [12]:
# Strict fit on development set only
preprocessor_candidate_a.fit(X_dev)
preprocessor_candidate_b.fit(X_dev)

X_dev_trans_a = preprocessor_candidate_a.transform(X_dev)
X_holdout_trans_a = preprocessor_candidate_a.transform(X_holdout)

X_dev_trans_b = preprocessor_candidate_b.transform(X_dev)
X_holdout_trans_b = preprocessor_candidate_b.transform(X_holdout)

# Verification checks
assert not np.isnan(X_dev_trans_a).any(), "NaN in X_dev_trans_a"
assert not np.isnan(X_holdout_trans_a).any(), "NaN in X_holdout_trans_a"
assert not np.isnan(X_dev_trans_b).any(), "NaN in X_dev_trans_b"
assert not np.isnan(X_holdout_trans_b).any(), "NaN in X_holdout_trans_b"

print(f"Candidate A Transformed Shape: X_dev={X_dev_trans_a.shape}, X_holdout={X_holdout_trans_a.shape}")
print(f"Candidate B Transformed Shape: X_dev={X_dev_trans_b.shape}, X_holdout={X_holdout_trans_b.shape}")
assert X_dev_trans_a.shape[1] == 56, f"Expected 56 features for Cand A, got {X_dev_trans_a.shape[1]}"
assert X_dev_trans_b.shape[1] == 59, f"Expected 59 features for Cand B, got {X_dev_trans_b.shape[1]}"
print("Transformed feature dimension counts verified: Candidate A = 56, Candidate B = 59.")

Candidate A Transformed Shape: X_dev=(242, 56), X_holdout=(61, 56)
Candidate B Transformed Shape: X_dev=(242, 59), X_holdout=(61, 59)
Transformed feature dimension counts verified: Candidate A = 56, Candidate B = 59.


## 13. Transformed Feature Audit & Output Schema
Audit exact transformed feature names for both candidate configurations.

In [13]:
features_a = preprocessor_candidate_a.get_feature_names_out()
features_b = preprocessor_candidate_b.get_feature_names_out()

print(f"Transformed Features Count Candidate A (Ordinal VHD): {len(features_a)}")
print(f"Transformed Features Count Candidate B (One-Hot VHD): {len(features_b)}")

df_feat_audit = pd.DataFrame({
    'Candidate A Index': range(len(features_a)),
    'Feature Name (Candidate A)': features_a
})
display(df_feat_audit.head(15))
display(df_feat_audit.tail(15))

Transformed Features Count Candidate A (Ordinal VHD): 56
Transformed Features Count Candidate B (One-Hot VHD): 59


,Candidate A Index,Feature Name (Candidate A)
0,0,num__Age
1,1,num__Weight
2,2,num__Length
3,3,num__BMI
4,4,num__BP
5,5,num__PR
6,6,num__FBS
7,7,num__CR
8,8,num__TG
9,9,num__LDL


,Candidate A Index,Feature Name (Candidate A)
41,41,bbb__BBB_N
42,42,bbb__BBB_LBBB
43,43,bbb__BBB_RBBB
44,44,vhd__VHD
45,45,bin_num__DM
46,46,bin_num__HTN
47,47,bin_num__Current Smoker
48,48,bin_num__EX-Smoker
49,49,bin_num__FH
50,50,bin_num__Edema


## 14. Target-Specific Execution Interface
Map each of the four target outcomes to its designated cross-validation splitter and clinical role.

In [14]:
TARGET_PIPELINES = {
    'Cath': {'role': 'Overall CAD Risk', 'cv_splitter': TARGET_CV_SPLITTERS['Cath']},
    'LAD':  {'role': 'Left Anterior Descending Stenosis', 'cv_splitter': TARGET_CV_SPLITTERS['LAD']},
    'LCX':  {'role': 'Left Circumflex Stenosis', 'cv_splitter': TARGET_CV_SPLITTERS['LCX']},
    'RCA':  {'role': 'Right Coronary Artery Stenosis', 'cv_splitter': TARGET_CV_SPLITTERS['RCA']}
}

for t, meta in TARGET_PIPELINES.items():
    print(f"Target: {t:5s} | Role: {meta['role']:40s} | Development Positives: {y_dev[t].sum():3d}/{len(y_dev)}")

Target: Cath  | Role: Overall CAD Risk                         | Development Positives: 173/242
Target: LAD   | Role: Left Anterior Descending Stenosis        | Development Positives: 143/242
Target: LCX   | Role: Left Circumflex Stenosis                 | Development Positives:  93/242
Target: RCA   | Role: Right Coronary Artery Stenosis           | Development Positives:  94/242


## 15. Deployment Consistency & Batch Invariance Check
Confirm that single-record inference (API intake) yields mathematically identical vectors to batch processing.

In [15]:
# Single patient record vs batch of records
single_record = X_dev.iloc[[0]]
batch_records = X_dev.iloc[:10]

x_single_a = preprocessor_candidate_a.transform(single_record)
x_batch_a = preprocessor_candidate_a.transform(batch_records)

assert np.allclose(x_single_a[0], x_batch_a[0]), "Single record does not match batch transformed output!"
print("Batch invariance test passed: single-patient inference output matches batch slice exactly.")

Batch invariance test passed: single-patient inference output matches batch slice exactly.


## 16. Save Preprocessing Metadata & Contract Configuration
Persist metadata, split indices, feature group schemas, and feature counts to preprocessing_config.json.

In [16]:
PREPROCESSING_CONFIG = {
    'schema_version': '1.0',
    'random_state': RANDOM_STATE,
    'cohort_size': len(df),
    'dev_size': len(X_dev),
    'holdout_size': len(X_holdout),
    'raw_feature_count': len(ALL_MODEL_FEATURES),
    'transformed_features_count': {
        'candidate_a_ordinal_vhd': 56,
        'candidate_b_onehot_vhd': 59
    },
    'dropped_features': DROP_COLUMNS,
    'target_columns': TARGET_COLUMNS,
    'target_mappings': TARGET_MAPPINGS,
    'feature_groups': {
        'all_numeric_features': ALL_NUMERIC_FEATURES,
        'binary_str_features': BINARY_STR_FEATURES,
        'binary_num_features': BINARY_NUM_FEATURES,
        'sex_feature': SEX_FEATURE,
        'nominal_features': NOMINAL_FEATURES,
        'ordinal_cat_features': ORDINAL_CAT_FEATURES
    },
    'cv_strategy': {
        'type': 'RepeatedStratifiedKFold',
        'n_splits': 5,
        'n_repeats': 5,
        'evaluations_per_target': 25,
        'stratification_mode': 'per_target_independent'
    },
    'holdout_indices': holdout_idx.tolist(),
    'train_indices': train_idx.tolist()
}

with open('preprocessing_config.json', 'w') as f:
    json.dump(PREPROCESSING_CONFIG, f, indent=4)

print("Saved updated preprocessing configuration to preprocessing_config.json")

Saved updated preprocessing configuration to preprocessing_config.json


## 17. Final Validation Contract
Formal specification of locked preprocessing and evaluation rules governing Notebooks 3 through 6.

In [17]:
validation_contract = pd.DataFrame([
    {'Component': 'Raw Dataset Cohort', 'Specification': 'N=303 patients, 59 raw columns'},
    {'Component': 'Deterministic Cleaning', 'Specification': 'Fmale -> Female, Exertional CP dropped (N=54 features)'},
    {'Component': 'Target Segregation', 'Specification': 'Cath, LAD, LCX, RCA excluded from feature matrices'},
    {'Component': 'Target Encoding', 'Specification': 'Binary 1/0 mapped deterministically'},
    {'Component': 'Train / Holdout Partition', 'Specification': '80% Dev (N=242), 20% Holdout (N=61) stratified on Cath'},
    {'Component': 'Holdout Governance', 'Specification': 'Untouched and sealed until locked final evaluation in Notebook 6'},
    {'Component': 'CV Strategy', 'Specification': 'Target-specific RepeatedStratifiedKFold (5x5, 25 evaluations per target)'},
    {'Component': 'Fold Preprocessing Isolation', 'Specification': 'Enforced strictly via sklearn Pipeline in Notebooks 3-5'},
    {'Component': 'Candidate A Preprocessing', 'Specification': 'Ordinal VHD -> exactly 56 transformed features'},
    {'Component': 'Candidate B Preprocessing', 'Specification': 'One-Hot VHD -> exactly 59 transformed features'},
    {'Component': 'Deployment Consistency', 'Specification': 'Batch invariance verified for single-patient API inference'},
    {'Component': 'Modeling & Tuning Status', 'Specification': 'Zero models trained; zero hyperparameters tuned'}
])
display(validation_contract)
print("Notebook 2 complete. Preprocessing and validation contract locked for Notebook 3 (Baseline Models).")

,Component,Specification
0,Raw Dataset Cohort,"N=303 patients, 59 raw columns"
1,Deterministic Cleaning,"Fmale -> Female, Exertional CP dropped (N=54 f..."
2,Target Segregation,"Cath, LAD, LCX, RCA excluded from feature matr..."
3,Target Encoding,Binary 1/0 mapped deterministically
4,Train / Holdout Partition,"80% Dev (N=242), 20% Holdout (N=61) stratified..."
5,Holdout Governance,Untouched and sealed until locked final evalua...
6,CV Strategy,"Target-specific RepeatedStratifiedKFold (5x5, ..."
7,Fold Preprocessing Isolation,Enforced strictly via sklearn Pipeline in Note...
8,Candidate A Preprocessing,Ordinal VHD -> exactly 56 transformed features
9,Candidate B Preprocessing,One-Hot VHD -> exactly 59 transformed features


Notebook 2 complete. Preprocessing and validation contract locked for Notebook 3 (Baseline Models).
